# Notebook 01: LLM Embedding 餘弦相似度量測

**實驗假說**：LLM Embedding 的 cosine similarity 無法區分「語義相似性」與「語義連結性」。

本 Notebook 使用  中文 Embedding 模型，
對  中的 20 個詞對計算餘弦相似度，並依象限分組。

In [1]:
# === 套件匯入 ===
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from scipy.spatial.distance import cosine
import json
from pathlib import Path

# 可再現性：固定 random seed
np.random.seed(42)
RANDOM_SEED = 42

/Users/yefangwong/Documents/GitHub.nosync/token-meaning-lab/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# === 載入中文 Embedding 模型 ===
# 模型: BAAI/bge-large-zh-v1.5
# 說明: 北航 BGE 系列，目前中文語義向量 MTEB 排行榜最強之一
# 首次執行會自動從 HuggingFace 下載（約 1.3GB）

MODEL_NAME = "BAAI/bge-large-zh-v1.5"
print(f"載入模型: {MODEL_NAME}")
model = SentenceTransformer(MODEL_NAME)
print("模型載入完成！")

載入模型: BAAI/bge-large-zh-v1.5
模型載入完成！


In [10]:
# === 載入 Pre-Registered 刺激詞對組 ===
stimulus_path = Path("../design/stimulus_set.csv")
df = pd.read_csv(stimulus_path)
print(f"載入 {len(df)} 個詞對")
print(f"象限分布:{df.groupby('quadrant').size()}")
df.head(20)

載入 20 個詞對
象限分布:quadrant
Q1    5
Q2    5
Q3    5
Q4    5
dtype: int64


,pair_id,word1,word2,quadrant,true_similarity,true_relatedness,rationale_zh
0,P01,醫生,護士,Q1,HIGH,LOW,同為醫療從業人員義原高度重疊；但無典型事件共現
1,P02,貓,狗,Q1,HIGH,LOW,同為哺乳類寵物；日常共現但非因果事件關係
2,P03,蘋果,橘子,Q1,HIGH,LOW,同為水果上位概念完全重合；無工具/場所因果連結
3,P04,鋼琴,吉他,Q1,HIGH,LOW,同為樂器；演奏場合偶有重疊但非強因果連結
4,P05,教師,教授,Q1,HIGH,LOW,同為教育工作者；職銜相似但非同一事件參與者
5,P06,醫生,手術刀,Q2,LOW,HIGH,工具-使用者關係(Agent-Instrument)；本體類別完全不同
6,P07,貓,老鼠,Q2,LOW,HIGH,捕食者-獵物關係(Agent-Target)；事件共現極強
7,P08,鋼琴,音樂廳,Q2,LOW,HIGH,樂器-場所關係(Instrument-Location)；強事件連結
8,P09,蘋果,果農,Q2,LOW,HIGH,產品-生產者關係(Product-Agent)；因果連結強
9,P10,書,圖書館,Q2,LOW,HIGH,物件-儲存場所關係(Object-Location)；連結強


In [5]:
# === 計算每個詞對的 Cosine Similarity ===
def compute_cosine_similarity(word1: str, word2: str, model) -> float:
    """使用 SentenceTransformer 計算兩詞的餘弦相似度"""
    emb1 = model.encode(word1, normalize_embeddings=True)
    emb2 = model.encode(word2, normalize_embeddings=True)
    # normalize_embeddings=True 時，dot product 即為 cosine similarity
    return float(np.dot(emb1, emb2))

print("計算中... (每個詞對各 encode 兩次)")
df["llm_cosine"] = df.apply(
    lambda row: compute_cosine_similarity(row["word1"], row["word2"], model),
    axis=1
)
print("完成！")
df[["pair_id", "word1", "word2", "quadrant", "llm_cosine"]]

計算中... (每個詞對各 encode 兩次)
完成！


,pair_id,word1,word2,quadrant,llm_cosine
0,P01,醫生,護士,Q1,0.719874
1,P02,貓,狗,Q1,0.683817
2,P03,蘋果,橘子,Q1,0.439955
3,P04,鋼琴,吉他,Q1,0.633672
4,P05,教師,教授,Q1,0.687674
5,P06,醫生,手術刀,Q2,0.499631
6,P07,貓,老鼠,Q2,0.615028
7,P08,鋼琴,音樂廳,Q2,0.442907
8,P09,蘋果,果農,Q2,0.546691
9,P10,書,圖書館,Q2,0.648320


In [8]:
# === 核心分析：各象限 cosine score 均值比較 ===
# 【關鍵預測】：若 H1 成立，Q1 與 Q2 的均值應無顯著差異

from scipy import stats

quadrant_summary = df.groupby("quadrant")["llm_cosine"].agg(["mean", "std", "min", "max"])
print("=== 各象限 LLM Cosine Similarity 統計 ===")
print(quadrant_summary.round(4))

# Q1 vs Q2 獨立樣本 t-test
q1_scores = df[df["quadrant"]=="Q1"]["llm_cosine"].values
q2_scores = df[df["quadrant"]=="Q2"]["llm_cosine"].values
t_stat, p_val = stats.ttest_ind(q1_scores, q2_scores)

print(f"Q1 均值: {q1_scores.mean():.4f}")
print(f"Q2 均值: {q2_scores.mean():.4f}")
print(f"t 統計量: {t_stat:.4f}")
print(f"p 值: {p_val:.4f}")
print(f"結論: {'H0 無法拒絕 → Q1≈Q2，LLM 無法區分相似性與連結性 ✓ 支持 H1' if p_val > 0.05 else 'H0 被拒絕 → LLM 能區分兩者 ✗ 不支持 H1'}")

=== 各象限 LLM Cosine Similarity 統計 ===
            mean     std     min     max
quadrant                                
Q1        0.6330  0.1122  0.4400  0.7199
Q2        0.5505  0.0835  0.4429  0.6483
Q3        0.6560  0.0559  0.5705  0.7092
Q4        0.3869  0.0435  0.3348  0.4525
Q1 均值: 0.6330
Q2 均值: 0.5505
t 統計量: 1.3183
p 值: 0.2239
結論: H0 無法拒絕 → Q1≈Q2，LLM 無法區分相似性與連結性 ✓ 支持 H1


In [9]:
# === 儲存結果供後續 Notebook 使用 ===
output_path = Path("../data/results/01_llm_cosine_scores.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_path, index=False)
print(f"結果已儲存至 {output_path}")

結果已儲存至 ../data/results/01_llm_cosine_scores.csv
